<a href="https://colab.research.google.com/github/priyal6/finetuning/blob/main/SFT%2BRL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
pip install torch transformers datasets trl accelerate

In [7]:
from transformers import AutoTokenizer

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
from datasets import Dataset

sft_data = Dataset.from_dict({
    "text": [
        "What is 2 + 3? 2 + 3 equals 5.",
        "What is 1 + 1? 1 + 1 equals 2."
    ]
})

In [9]:
def tokenize_sft(example):
  tokens = tokenizer(
      example["text"],
      truncation=True,
      padding="max_length",
      max_length=64,
  )

  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized_sft = sft_data.map(tokenize_sft, remove_columns=['text'])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [10]:
from transformers import(
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)

sft_model = AutoModelForCausalLM.from_pretrained(model_name)

sft_args = TrainingArguments(
    output_dir="./sft_model",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=10,
    report_to="none"
)

sft_trainer = Trainer(
    model=sft_model,
    args=sft_args,
    train_dataset=tokenized_sft,
    tokenizer=tokenizer,
)

sft_trainer.train()
sft_trainer.save_model("./sft_model")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/tmp/ipython-input-3372552522.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  sft_trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,8.065300
2,6.034700
3,3.937900


In [11]:
#reward model

rm_data = Dataset.from_dict({
    "prompt": ["What is 2 + 3?"],
    "chosen": ["2 + 3 equals 5."],
    "rejected": ["I think it is 6."]
})

In [12]:
def tokenize_rm(example):
  chosen = tokenizer(
      example["prompt"] + " " + example["chosen"],
      truncation = True,
      padding = "max_length",
      max_length = 64,
  )

  rejected = tokenizer(
      example['prompt'] + " " + example["rejected"],
      truncation = True,
      padding = "max_length",
      max_length = 64,
  )

  return {
        "input_ids_chosen": chosen["input_ids"],
        "attention_mask_chosen": chosen["attention_mask"],
        "input_ids_rejected": rejected["input_ids"],
        "attention_mask_rejected": rejected["attention_mask"],
    }

tokenized_rm = rm_data.map(tokenize_rm, remove_columns= rm_data.column_names)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [13]:
from transformers import AutoModelForSequenceClassification

reward_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
import torch

def reward_data_collator(features):
    return {
        "input_ids_chosen": torch.tensor(
            [f["input_ids_chosen"] for f in features]
        ),
        "attention_mask_chosen": torch.tensor(
            [f["attention_mask_chosen"] for f in features]
        ),
        "input_ids_rejected": torch.tensor(
            [f["input_ids_rejected"] for f in features]
        ),
        "attention_mask_rejected": torch.tensor(
            [f["attention_mask_rejected"] for f in features]
        ),
    }


In [15]:
import torch
from transformers import Trainer

class RewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        r_chosen = model(
            input_ids=inputs["input_ids_chosen"],
            attention_mask=inputs["attention_mask_chosen"],
        ).logits

        r_rejected = model(
            input_ids=inputs["input_ids_rejected"],
            attention_mask=inputs["attention_mask_rejected"],
        ).logits

        loss = -torch.log(torch.sigmoid(r_chosen - r_rejected)).mean()
        return (loss, None) if return_outputs else loss

In [16]:
rm_args = TrainingArguments(
    output_dir="./reward_model",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=10,
    report_to="none",
    remove_unused_columns=False,  # ← THIS IS THE FIX
)


rm_trainer = RewardTrainer(
    model=reward_model,
    args=rm_args,
    train_dataset=tokenized_rm,
    data_collator = reward_data_collator
)

rm_trainer.train()
rm_trainer.save_model("./reward_model")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.469300
2,0.339000
3,1.460300


In [17]:
#ppo
!pip install trl
from trl import PPOTrainer, PPOConfig
from transformers import AutoModelForCausalLM

In [36]:
policy_model = AutoModelForCausalLM.from_pretrained("sft_model", local_files_only=True)
ref_model = AutoModelForCausalLM.from_pretrained("sft_model", local_files_only=True)
reward_model = AutoModelForSequenceClassification.from_pretrained("reward_model", local_files_only=True)
value_model = AutoModelForCausalLM.from_pretrained("sft_model", local_files_only=True) # Initialize value model

In [33]:
from datasets import Dataset

ppo_dataset = Dataset.from_dict({
    "prompt": [
        "What is 2 + 3?",
        "What is 1 + 1?",
    ]
})


In [38]:
from trl import PPOTrainer, PPOConfig

ppo_config = PPOConfig(
    batch_size=1,
    learning_rate=1e-5,
    bf16=False,
    fp16=False,
)

ppo_trainer = PPOTrainer(
    ppo_config,    # 1. config (PPOConfig object)
    tokenizer,     # 2. processing_class (tokenizer object)
    policy_model,  # 3. model (policy model)
    ref_model,     # 4. ref_model (reference model)
    reward_model,  # 5. reward_model (reward model)
    ppo_dataset,   # 6. train_dataset (PPO training dataset)
    value_model    # 7. value_model (value model)
)

<string>:167: FutureWarning: The `PPOConfig` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import PPOConfig`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
/tmp/ipython-input-2869354908.py:10: FutureWarning: The `PPOTrainer` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import PPOTrainer`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
  ppo_trainer = PPOTrainer(


In [39]:
prompt = "What is 2 + 3?"
query = tokenizer(prompt, return_tensors="pt").input_ids

response = policy_model.generate(query, max_new_tokens=20)
response_text = tokenizer.decode(response[0], skip_special_tokens=True)

reward_inputs = tokenizer(
    prompt + " " + response_text,
    return_tensors="pt",
    truncation=True,
)
reward = reward_model(**reward_inputs).logits.squeeze()

ppo_trainer.step(
    queries=[query[0]],
    responses=[response[0]],
    rewards=[reward.detach()],
)


AttributeError: 'PPOTrainer' object has no attribute 'step'